In [2]:
# import libraries
from pydantic import BaseModel
from openai import AsyncOpenAI
import asyncio
from dotenv import load_dotenv
import random
import os
import json
from sklearn.metrics import classification_report

In [3]:
# initialize empty dataset list
validation_data = []

# load the validation data
with open("../../../01_data/classification/training_validation_sets/stance/validation_set.json", "r") as f:
    raw_val_data = json.load(f)

# extract only the original sentence without any augmentations
tag_to_label = {
    "pos": "positive",
    "neutral": "neutral",
    "neg": "negative"
}

for task in raw_val_data:
    sentence = task[0]["sentence"]
    social_group = task[0]["group"]
    stance = tag_to_label[task[0]["stance"]]
    validation_data.append({"sentence": sentence,
                            "social_group": social_group,
                            "stance": stance})

In [4]:
# compile prompt templates
medium = """
## Task Objective
Label the following sentence according to the stance the speaker expresses towards the highlighted social group. Return this label as a JSON object.
The stance can be either positive, neutral or negative. Label the sentence according to the following definition and criteria.

## Positive Stance
The text is positive towards the group if it expresses some sort of support or positive affect.
Especially within questions, a positive stance can also be expressed indirectly by raising the interests of the respective group or criticizing their disadvantage.

## Neutral Stance
The text is neutral towards the group if it references the group, but neither a positive nor negative stance are taken.
This happens mostly when the speaker mentions the group by stating a fact or when the group is part of a title or organization name.

## Negative Stance
The text is negative towards the group if it expresses any form of critical or negative feeling towards the mentioned group. Raising awareness of the disadvantage faced by a group is not a negative but a positive stance.
"""

short = """
## Task Objective
Label the following sentence according to the stance the speaker expresses towards the highlighted social group. Return this label as a JSON object.
The stance can be either positive, neutral or negative.

## Positive Stance
The sentence expesses support or positive affect towards the group. Especially within questions, a positive stance can also be expressed indirectly by raising the interests of the group.

## Neutral Stance
The sentence references the group, but neither a positive nor negative stance are taken. This happens mostly within factual statements.

## Negative Stance
The sentence expresses a critical or negative feeling towards the mentioned group. Raising awareness of the disadvantage faced by a group is not a negative but a positive stance.
"""


# compile manual few-shot examples in json format
positive_example = {"text": "Stance towards young people in: Our party stands for improving the job opportunities of young people.",
        "llm_text": '{"stance": "positive"}'}
        
subtle_positive_example_1 = {"text": "Stance towards pupils: What is the Government's approach on creating a good working environment for pupils and teachers in our schools?",
     "llm_text": '{"stance": "positive"}'}

subtle_positive_example_2 = {"text": "Stance towards women: Women experience discrimination repeatedly throughout their lives",
     "llm_text": '{"stance": "positive"}'}

neutral_example = {"text": "Stance towards GP: Each person in this country should have the chance to get an appointment at his or her GP within a couple of days.",
     "llm_text": '{"stance": "neutral"}'}

negative_example = {"text": "Stance towards criminal offenders: We must do everything we can to tackle violence on our streets and get criminal offenders into jail.",
     "llm_text": '{"stance": "negative"}'}

# collect all in a list and mix up the order
few_shot_examples_all = [positive_example, subtle_positive_example_1, subtle_positive_example_2, neutral_example, negative_example]

In [5]:
# create prompt template
def compile_prompt_stance(system_prompt, few_shot_examples, num_positive_examples, test_sentence, social_group):

        chat = [
                {
                        "role": "system",
                        "content": system_prompt
                }
        ]

        # add all positive few-shot examples
        for i in range(num_positive_examples):
                chat.append({"role": "user", "content": f"Sentence: {few_shot_examples[i]['text']}"})
                chat.append({"role": "assistant", "content": few_shot_examples[i]["llm_text"]})

        # add the remaining few-shot examples
        chat.append({"role": "user", "content": f"Sentence: {few_shot_examples[-2]['text']}"})
        chat.append({"role": "assistant", "content": few_shot_examples[-2]["llm_text"]})
        chat.append({"role": "user", "content": f"Sentence: {few_shot_examples[-1]['text']}"})
        chat.append({"role": "assistant", "content": few_shot_examples[-1]["llm_text"]})    
        
        # add the test sentence
        chat.append({"role": "user", "content": f"Stance towards {social_group} in: {test_sentence}"})

        return chat

In [6]:
# create dictionary storing rate limits
model_limits = {
    "gpt-4o-mini":
    {"token_limit": 200000,
     "request_limit": 500
    },
    "gpt-4o":
    {"token_limit": 30000,
     "request_limit": 500
    },
    "gpt-5-nano":
    {"token_limit": 200000,
     "request_limit": 500
    }
    }

# function to send out the request
async def send_request(client, model_name, prompt, output_class, temp=None, reasoning_effort=None):

    if model_name == "gpt-5-nano" :                                   
        response = await client.responses.parse(model=model_name,
                                                input=prompt,
                                                text_format=output_class,
                                                reasoning = {"effort": reasoning_effort}
                                                )
    elif model_name == "gpt-4o-mini":
        response = await client.responses.parse(model=model_name,
                                                input=prompt,
                                                text_format=output_class,
                                                temperature=temp
                                                )

    stance = response.output_parsed.stance
    try:
        if not isinstance(stance, str):
            return None
        return stance
    except Exception:
        return None
                                                                               
async def dispatch_all(client, model_name, system_message, few_shot_examples, num_positive_examples,
                       output_class, validation_data, temp, reasoning_effort, safe_interval):
    tasks = []
    for row in validation_data:
        sentence = row["sentence"]
        social_group = row["social_group"]
        prompt = compile_prompt_stance(
            system_message,
            few_shot_examples,
            num_positive_examples,
            sentence,
            social_group
        )
        # create a task and fire it, do not wait
        task = asyncio.create_task(send_request(client, model_name, prompt, output_class, temp, reasoning_effort))
        tasks.append(task)
        
        # wait before starting the next request
        await asyncio.sleep(safe_interval)

    # gather all results once everything is started
    return await asyncio.gather(*tasks)

In [23]:
# select the model, system message and number of few shot examples
model_name = "gpt-5-nano"
system_message = medium

# set temperature and reasoning effort
temp = 0
reasoning_effort = "medium"

# empirically test a safe rate per minute
if model_name == "gpt-4o":
    safe_rpm = 50
else:
    safe_rpm = 100

# calculate a safe interval in which requests are sent
safe_interval = 60.0 / safe_rpm

# define class for the output
class StanceJSON(BaseModel):
    stance: str

# create client for interacting with API
load_dotenv()
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# get random indices
random.seed(0)
random_indices = random.sample(range(len(validation_data)), 100)
val_subset = [validation_data[i] for i in random_indices]
llm_output = await dispatch_all(client, model_name, system_message, few_shot_examples_all, 3, StanceJSON, val_subset, temp, reasoning_effort, safe_interval)

In [24]:
for idx in range(0, len(val_subset)):
    print(val_subset[idx]["sentence"])
    print(val_subset[idx]["social_group"])
    print(f"Ground truth: {val_subset[idx]["stance"]}")
    print(f"LLM prediction: {llm_output[idx]}")
    print("-"*50)

Very recently, the Government reduced the waiting time for gay people who want to give blood from 12 months of celibacy to three months, which was welcomed by the LGBT community.
gay people
Ground truth: positive
LLM prediction: positive
--------------------------------------------------
We must go into schools and teach our young people that domestic abuse, be it physical, mental or sexual, is totally unacceptable.
young people
Ground truth: neutral
LLM prediction: positive
--------------------------------------------------
Shelter's rent watch report 2011 found that, on average, private rents in 55% of local authorities in England were unaffordable for ordinary working families, and that 38% of privately renting families with children had to cut down on food to pay their rent.
privately renting families with children
Ground truth: neutral
LLM prediction: positive
--------------------------------------------------
Of course we benefit from the activities of American pilots in Afghanis

In [25]:
# inspect the results
results = {}
ground_truth = [item["stance"] for item in val_subset]
prediction = [stance for stance in llm_output]
metrics = classification_report(ground_truth, prediction, output_dict=True)
results["gpt-5-nano"] = {
           "negative_f1": metrics["negative"]["f1-score"],
           "neutral_f1": metrics["neutral"]["f1-score"],
           "positive_f1": metrics["positive"]["f1-score"],
           "macro_f1": metrics["macro avg"]["f1-score"]
           }
results

{'gpt-5-nano': {'negative_f1': 0.46153846153846156,
  'neutral_f1': 0.576271186440678,
  'positive_f1': 0.796875,
  'macro_f1': 0.6115615493263798}}

In [7]:
# set up temperatures and top
temperatures = [0, 0.25, 0.5]
reasoning_efforts = [None, "low", "medium", "high"]

# create client for interacting with API
load_dotenv()
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# define class for the output
class StanceJSON(BaseModel):
    stance: str

# tune temperature value for 4o-mini model
model_name = "gpt-4o-mini"
system_message = short
num_pos_examples = 2
safe_rpm = 1000
safe_interval = 60.0 / safe_rpm
results_4o_mini = {}
print("Tuning of 4o-mini starts")
print("-"*50)
for temp in temperatures:
    llm_output = await dispatch_all(client, model_name, system_message, few_shot_examples_all, num_pos_examples, StanceJSON, validation_data, temp, None, safe_interval)
    ground_truth = [item["stance"] for item in validation_data]
    prediction = [stance for stance in llm_output]
    report = classification_report(ground_truth, prediction, output_dict=True)
    macro_f1 = report["macro avg"]["f1-score"]
    print(f"Macro F1 for temperature {temp}: {macro_f1}")
    results_4o_mini[temp] = macro_f1

# test all reasoning effort values for 5-nano
model_name = "gpt-5-nano"
system_message = medium
num_pos_examples = 3
safe_rpm = 750
safe_interval = 60.0 / safe_rpm
results_5_nano = {}
print("-"*50)
print("Tuning of 5-nano starts")
print("-"*50)
for reasoning_effort in reasoning_efforts:
    llm_output = await dispatch_all(client, model_name, system_message, few_shot_examples_all, num_pos_examples, StanceJSON, validation_data, None, reasoning_effort, safe_interval)
    ground_truth = [item["stance"] for item in validation_data]
    prediction = [stance for stance in llm_output]
    report = classification_report(ground_truth, prediction, output_dict=True)
    macro_f1 = report["macro avg"]["f1-score"]
    print(f"Macro F1 for temperature {reasoning_effort}: {macro_f1}")
    results_5_nano[reasoning_effort] = macro_f1

Tuning of 4o-mini starts
--------------------------------------------------
Macro F1 for temperature 0: 0.5441158053918423
Macro F1 for temperature 0.25: 0.5398148349971664
Macro F1 for temperature 0.5: 0.5442916537265995
--------------------------------------------------
Tuning of 5-nano starts
--------------------------------------------------
Macro F1 for temperature None: 0.691995584427961
Macro F1 for temperature low: 0.6665406276278647
Macro F1 for temperature medium: 0.680687044476539
Macro F1 for temperature high: 0.6836830265635112


In [8]:
hyperparameter_tuning_results = [results_4o_mini, results_5_nano]
with open("hyperparameter_tuning_results/ht_genllm_stance.json", "w") as f:
    json.dump(hyperparameter_tuning_results, f)